# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamzaviour/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Lane:** Content Opportunity Scoring. **Data:** 30k-row starter snapshot (trailing 90 days, one row per page, 32 clients). This is a practical, **non-production** playbook: it turns the validated Week-6 output into a ranked, human-reviewed queue with reason codes, limits and monitoring rules. All wording is observed / measured / directional / decision-support.

In [1]:
import os, json, warnings, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
if os.path.exists("../../data/raw/content_refresh_anonymized.csv"): os.chdir("../..")
os.makedirs("work/outputs", exist_ok=True); os.makedirs("work/figures", exist_ok=True)
PATH = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(PATH):
    import urllib.request
    PATH = "content_refresh_anonymized.csv"
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Hamzaviour/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv", PATH)
df = pd.read_csv(PATH)
df["declining"] = (df.trend_direction == "down").astype(int)   # used ONLY to train/evaluate, never as a rule input
y, SEED = df.declining.values, 42
print(df.shape, "base rate declining:", round(y.mean(), 3))

(30000, 45) base rate declining: 0.542


## 0. The score behind the playbook (from Week 6)

I reuse the **strict-timeline logistic regression** from the Week-6 audit: static metadata plus the *earliest* 30 days of volume (days 61-90 back), never the 30-day windows the label is built from. Scores are **out-of-fold, grouped by client**, so every page is scored by a model that never saw its client. Its measured skill is modest (AUC about 0.66), so it is used only to *order a human review queue*.

In [2]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

for c in ["impressions", "clicks", "sessions"]:
    df[f"{c}_earliest30"] = (df[f"{c}_90d"] - df[f"{c}_last_30d"] - df[f"{c}_prev_30d"]).clip(lower=0)
num = ["search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days", "days_since_last_update",
       "impressions_earliest30", "clicks_earliest30", "sessions_earliest30"]
cat = ["content_type", "main_intent", "competition_level"]
X = df[num + cat].copy()
for c in ["search_volume", "word_count"]: X["has_" + c] = X[c].notna().astype(int)
for c in ["impressions_earliest30", "clicks_earliest30", "sessions_earliest30"]: X[c] = np.log1p(X[c])
X = pd.get_dummies(X, columns=cat, dummy_na=True)

folds = list(GroupKFold(5).split(X, y, df.client_id))
p = np.zeros(len(df)); fold_p50 = []
for tr, te in folds:
    m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000, C=0.5)).fit(X.iloc[tr], y[tr])
    p[te] = m.predict_proba(X.iloc[te])[:, 1]
    top = np.argsort(-p[te], kind="stable")[:50]; fold_p50.append(y[te][top].mean())
df["p_decline"] = p
print(f"out-of-fold AUC {roc_auc_score(y, p):.3f}; per-fold P@50: {np.round(fold_p50, 2)} vs base rate {y.mean():.2f}")

out-of-fold AUC 0.650; per-fold P@50: [0.8  0.9  0.66 0.92 0.86] vs base rate 0.54


## 1. Ranked actions + reason codes

One page gets **one action and one reason code**. Each rule estimates *clicks at stake per 30 days* so different rules can be ranked on one scale:

| action | reason code | rule (transparent, no fitted weights) | clicks at stake (30d) |
|---|---|---|---|
| `rewrite_title_meta` | `ctr_below_position_norm` | position 0-20, ≥ 1,000 impressions, CTR below the pooled CTR of its position band (Week-4 rule) | missed clicks over 90d ÷ 3 |
| `refresh_content` | `stale_and_likely_declining` | model p(decline) ≥ 0.60 and ≥ 90 days since update | 90d clicks ÷ 3 × 0.20 × p(decline) |
| `protect_page_one` | `page_one_at_risk` | position ≤ 10, ≥ 1,000 impressions, p(decline) ≥ 0.60, updated < 90 days ago | 90d clicks ÷ 3 × 0.20 × p(decline) |
| `no_action_low_data` | `too_little_data` | fewer than 200 impressions in 90 days, or tiny-volume archetype | 0 |
| `monitor` | `no_rule_fired` | everything else | 0 |

If more than one rule fires, the **highest clicks-at-stake** wins. Pages are then ranked *within* each action (section 5 explains why, and which key is used for each). The 0.20 comes from the label definition (a "decline" is a drop of more than 20%), so it is a floor on the size of a decline, not a measured average. Rule inputs are 90-day figures (position, CTR, impressions), which is a disclosed limit of this snapshot.

In [3]:
pos_ok = (df.avg_position > 0) & (df.avg_position <= 20)
d = df[pos_ok]; BAND = lambda s: pd.cut(s, [0, 3, 10, 20], labels=["1-3", "3-10", "10-20"])
g = d.assign(b=BAND(d.avg_position)).groupby("b", observed=True)
norm = g.clicks_90d.sum() / g.impressions_90d.sum() * 100
exp_ctr = BAND(df.avg_position.where(pos_ok)).map(norm).astype(float)

ctr_elig = pos_ok & (df.impressions_90d >= 1000)
s_ctr = np.where(ctr_elig, df.impressions_90d * (exp_ctr - df.ctr).clip(lower=0) / 100 / 3, 0.0)
risk = df.clicks_90d / 3 * 0.20 * df.p_decline
hi = df.p_decline >= 0.60
s_ref = np.where(hi & (df.days_since_last_update >= 90), risk, 0.0)
s_prot = np.where(hi & (df.avg_position > 0) & (df.avg_position <= 10) & (df.impressions_90d >= 1000)
                  & (df.days_since_last_update < 90), risk, 0.0)

S = pd.DataFrame({"rewrite_title_meta": s_ctr, "refresh_content": s_ref, "protect_page_one": s_prot}, index=df.index)
CODE = {"rewrite_title_meta": "ctr_below_position_norm", "refresh_content": "stale_and_likely_declining",
        "protect_page_one": "page_one_at_risk", "no_action_low_data": "too_little_data", "monitor": "no_rule_fired"}
df["clicks_at_stake_30d"] = S.max(axis=1)
df["action"] = np.where(df.clicks_at_stake_30d > 0, S.idxmax(axis=1), "monitor")
df.loc[df.impressions_90d < 200, ["action", "clicks_at_stake_30d"]] = ["no_action_low_data", 0.0]
df["reason_code"] = df.action.map(CODE)
print(df.groupby("action").agg(n=("action", "size"), clicks_at_stake_30d=("clicks_at_stake_30d", "sum"),
                               actual_decline_rate=("declining", "mean")).round(2).to_string())
print("\nbase rate of decline:", round(y.mean(), 3))

                        n  clicks_at_stake_30d  actual_decline_rate
action                                                             
monitor             10114                 0.00                 0.58
no_action_low_data   9914                 0.00                 0.43
protect_page_one      703              1241.98                 0.53
refresh_content      2695              2666.76                 0.60
rewrite_title_meta   6574             45133.18                 0.63

base rate of decline: 0.542


**Reading:** the table shows how many pages each action covers and the observed decline rate among them next to the base rate. Where an action's decline rate is not above the base rate, that action is not backed by the decline label (the CTR rule targets missed clicks, not decline, so that is expected for it).

## 2. Archetypes → actions

K-Means (k = 4, standardised log-impressions, position, CTR, age, word count, days since update). I name each cluster **after** reading its medians. Cluster separation is weak (silhouette about 0.28), so archetypes are a reading aid, not hard categories.

In [4]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
A = pd.DataFrame({"log_impr": np.log1p(df.impressions_90d),
                  "position": df.avg_position.where(df.avg_position > 0).fillna(df.avg_position[df.avg_position > 0].median()),
                  "ctr": df.ctr, "age": df.content_age_days,
                  "words": df.word_count.fillna(df.word_count.median()), "days_since_update": df.days_since_last_update})
Z = StandardScaler().fit_transform(A)
km = KMeans(4, n_init=10, random_state=SEED).fit(Z)
print("silhouette:", round(silhouette_score(Z, km.labels_, sample_size=5000, random_state=1), 3))
A["c"] = km.labels_; A["declining"] = y
prof = A.groupby("c").agg(n=("c", "size"), log_impr=("log_impr", "median"), position=("position", "median"), ctr=("ctr", "median"),
                          age=("age", "median"), days_since_update=("days_since_update", "median"), declining=("declining", "mean")).round(2)

# name AFTER inspecting medians: tiny cluster, stalest, youngest, rest
names = {}
names[prof.n.idxmin()] = "tiny_volume_outliers"
rest = prof.drop(index=prof.n.idxmin())
names[rest.days_since_update.idxmax()] = "stale_but_visible"
rest = rest.drop(index=rest.days_since_update.idxmax())
names[rest.age.idxmin()] = "young_and_volatile"
names[[i for i in prof.index if i not in names][0]] = "established_steady"
prof["archetype"] = prof.index.map(names); df["archetype"] = A.c.map(names).values
print(prof.set_index("archetype").to_string())

silhouette: 0.284
                          n  log_impr  position    ctr    age  days_since_update  declining
archetype                                                                                  
stale_but_visible      9195      7.44      13.5   0.11  258.0              104.0       0.61
young_and_volatile    11809      6.21      10.1   0.06  125.0               20.0       0.61
established_steady     8836      6.08      11.5   0.00  441.0               20.0       0.39
tiny_volume_outliers    160      1.39       4.0  33.33  296.5               20.0       0.16


In [5]:
MAP = pd.DataFrame([
    ("stale_but_visible",    "refresh_content",     "visible, not updated for ~3+ months, highest observed decline rate with the young group"),
    ("young_and_volatile",   "monitor",             "recent pages swing a lot; wait one more cycle before editing"),
    ("established_steady",   "rewrite_title_meta",  "older, recently updated, lowest observed decline rate; the value here is CTR, not rescue"),
    ("tiny_volume_outliers", "no_action_low_data",  "very few pages with extreme CTR on tiny volume; fix data/measurement first"),
], columns=["archetype", "default_action", "why"])
print(MAP.to_string(index=False))
print("\nhow page-level actions fall across archetypes (counts):")
print(pd.crosstab(df.archetype, df.action).to_string())

           archetype     default_action                                                                                      why
   stale_but_visible    refresh_content  visible, not updated for ~3+ months, highest observed decline rate with the young group
  young_and_volatile            monitor                             recent pages swing a lot; wait one more cycle before editing
  established_steady rewrite_title_meta older, recently updated, lowest observed decline rate; the value here is CTR, not rescue
tiny_volume_outliers no_action_low_data               very few pages with extreme CTR on tiny volume; fix data/measurement first

how page-level actions fall across archetypes (counts):
action                monitor  no_action_low_data  protect_page_one  refresh_content  rewrite_title_meta
archetype                                                                                               
established_steady       3268                3579                32                2    

The archetype's `default_action` is the starting expectation; the **page-level rule in section 1 decides the actual action and reason code**. A page whose rule disagrees with its archetype goes to human review (section 4).

## 3. The decay / refresh insight

In [6]:
df["stale_bucket"] = pd.cut(df.days_since_last_update, [-1, 30, 90, 180, 365, 10000], labels=["0-30", "31-90", "91-180", "181-365", "365+"])
dec = df.groupby("stale_bucket", observed=True).agg(n=("declining", "size"), decline_rate=("declining", "mean")).round(3)
print(dec.to_string())
df["age_bucket"] = pd.cut(df.content_age_days, [89, 180, 365, 730, 10000], labels=["90-180", "181-365", "366-730", "730+"])
agb = df.groupby("age_bucket", observed=True).agg(n=("declining", "size"), decline_rate=("declining", "mean")).round(3)
print(); print(agb.to_string())

fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for a_, tab, ttl in [(ax[0], dec, "days since last update"), (ax[1], agb, "content age (days)")]:
    a_.bar(tab.index.astype(str), tab.decline_rate * 100, color="#3b6ea5")
    a_.axhline(y.mean() * 100, color="#b33", ls="--", lw=1, label=f"overall {y.mean()*100:.0f}%")
    for i, (r, n) in enumerate(zip(tab.decline_rate, tab.n)): a_.text(i, r * 100 + 1, f"n={n:,}", ha="center", fontsize=7)
    a_.set_title(ttl, fontsize=9); a_.set_ylabel("% pages declining"); a_.set_ylim(0, 80); a_.legend(fontsize=7)
fig.suptitle("Observed share of pages with a >20% impression drop (last 30d vs prior 30d)", fontsize=9)
fig.tight_layout(); fig.savefig("work/figures/decay_by_staleness_and_age.png", dpi=150); plt.close(fig)

                  n  decline_rate
stale_bucket                     
0-30          20480         0.511
31-90           175         0.589
91-180         9171         0.611
181-365         169         0.467
365+              5         0.600

                n  decline_rate
age_bucket                     
90-180      12272         0.627
181-365     11368         0.515
366-730      6360         0.426


**What the decay data shows (observed, this snapshot; figure saved to `work/figures/decay_by_staleness_and_age.png`):**

- **Staleness.** Pages not updated for 91-180 days declined more often (61%, n = 9,171) than pages updated in the last 30 days (51%, n = 20,480), against an overall 54%. The 31-90, 181-365 and 365+ buckets have only 175, 169 and 5 pages, so I do not read anything into them. The direction is consistent with "refresh visible pages before they decay", but the gap is about 10 percentage points and is observational: staleness and age move together here, so I cannot separate them.
- **Age goes the other way from what I expected.** Younger pages (90-180 days) declined more often (63%, n = 12,272) than older pages (366-730 days, 43%, n = 6,360). This differs from the paper's Finding #1, where declining pages were older on average; the slices, windows and thresholds differ (this one uses a ±20% label on active pages aged 90+ days), so I treat it as a reminder that "older = decaying" is not safe to assume, not as a contradiction.
- **Practical reading.** A decline flag on a young page may be post-launch settling, which is why the `young_and_volatile` archetype defaults to `monitor` rather than an edit. The refresh insight that I would act on, with human review, is narrow: *visible pages that have gone about 3-6 months without an update.*


## 4. Intended use, human review, and what must NOT be automated

**Intended use.** A weekly or monthly *review queue* for a content team: "look at these pages first, for this stated reason". It is decision-support. It orders human attention; it does not decide what to publish, delete or rewrite.

**Limits (measured or disclosed).**
- One 90-day snapshot, 32 clients, active pages only (≥ 1 impression, age ≥ 90 days). No time-ordered test was run; results may not hold for a new month or an unseen kind of client.
- Out-of-fold AUC about 0.66 with fold spread of 0.03-0.07; modest signal.
- Rule inputs (position, CTR, impressions) are 90-day figures that overlap the 30-day label window.
- Clicks at stake is an estimate built from stated assumptions, not a forecast. The decline label includes mean-reversion on tiny-volume pages.
- Correlation only: nothing here shows that an action *causes* recovery.

**Human-review rules.** A reviewer checks every queued page before any change:
1. Open the page and its query list; confirm the reason code makes sense on the live page.
2. Check the top-3 rows per client together: if one client dominates, suspect a site-wide cause (template, tracking change) before editing pages.
3. For `rewrite_title_meta`: check for SERP features (snippet, ads, video) that explain a low CTR.
4. For `refresh_content`: confirm the page is still wanted (topic, intent) before spending time.
5. If the rule's action disagrees with the archetype default, the reviewer decides; log the reason.
6. Record the decision (accept / reject / defer + one line why); rejected rows feed the monitoring check below.

**No-go list: do not automate**
- Publishing, deleting, redirecting or de-indexing any page from a score.
- Bulk rewriting titles or content with a script or an LLM without a person reading each result.
- Acting on `no_action_low_data` or `monitor` rows as if they were findings.
- Reporting "clicks at stake" as revenue or promised uplift.
- Applying the queue to a new client or a new month without re-running the checks below.
- Using the score to compare or rank *clients* or *writers*.

## 5. Cost / value thinking

Review time is an **assumption** (set below, change it to your team's reality), not a measurement. The measured part is how many clicks at stake the top of the queue holds compared with picking pages at random.

In [7]:
REVIEW_MIN_PER_PAGE = 15      # ASSUMPTION: minutes for a human to open, check and decide on one page
QUOTA = {"rewrite_title_meta": 20, "refresh_content": 20, "protect_page_one": 10}   # one sprint = 50 pages
K = sum(QUOTA.values())
act = df[df.action.isin(QUOTA)]
# Why quotas: on one common "clicks at stake" scale the CTR rule's estimates are much larger than the
# decline-risk estimates, so a single ranked list would be ~100% rewrite_title_meta and hide refresh/protect.
print("share of the single-scale top-50 that is rewrite_title_meta:",
      (act.nlargest(50, "clicks_at_stake_30d").action == "rewrite_title_meta").mean())
topk_size = pd.concat([act[act.action == a_].nlargest(n, "clicks_at_stake_30d") for a_, n in QUOTA.items()])
# Second ordering: for the two decline-based actions, rank by the model probability instead of by page size.
ORDER_KEY = {"rewrite_title_meta": "clicks_at_stake_30d", "refresh_content": "p_decline", "protect_page_one": "p_decline"}
topk = pd.concat([act[act.action == a_].nlargest(n, ORDER_KEY[a_]) for a_, n in QUOTA.items()])
cmp_ = pd.DataFrame({"ranked by clicks at stake (size)": topk_size.groupby("action").declining.mean(),
                     "ranked by p_decline (refresh/protect)": topk.groupby("action").declining.mean()}).round(2)
print("observed decline rate in the 50-page queue, two orderings (rewrite_title_meta uses size in both):")
print(cmp_.to_string()); print()
rng = np.random.default_rng(SEED)
seeds = [int(v) for v in rng.integers(0, 10**6, 200)]
rand_decl = np.mean([pd.concat([act[act.action == a_].sample(n, random_state=sd) for a_, n in QUOTA.items()]).declining.mean() for sd in seeds])
print(f"\nsprint queue = {K} pages {QUOTA}; review cost {K} x {REVIEW_MIN_PER_PAGE} min = {K*REVIEW_MIN_PER_PAGE/60:.1f} h (assumption)")
print(f"est. clicks at stake in the queue: {topk.clicks_at_stake_30d.sum():,.0f}/30d  ({topk.clicks_at_stake_30d.sum()/(K*REVIEW_MIN_PER_PAGE/60):,.0f} per review-hour)")
print("   (largest scores by construction, so this is a size estimate, not a validation)")
print("\nMEASURED check against the decline label (the label is not an input to the rules):")
chk = topk.groupby("action").declining.agg(["size", "mean"]).round(2)
chk["random draw, same action"] = [round(float(np.mean([act[act.action == a_].sample(QUOTA[a_], random_state=sd).declining.mean() for sd in seeds])), 2) for a_ in chk.index]
print(chk.to_string())
print("overall base rate:", round(y.mean(), 3), "| queue decline rate:", round(topk.declining.mean(), 2), "| random same-quota draw:", round(rand_decl, 2))


share of the single-scale top-50 that is rewrite_title_meta: 1.0
observed decline rate in the 50-page queue, two orderings (rewrite_title_meta uses size in both):
                    ranked by clicks at stake (size)  ranked by p_decline (refresh/protect)
action                                                                                     
protect_page_one                                0.10                                   0.40
refresh_content                                 0.25                                   0.85
rewrite_title_meta                              0.45                                   0.45




sprint queue = 50 pages {'rewrite_title_meta': 20, 'refresh_content': 20, 'protect_page_one': 10}; review cost 50 x 15 min = 12.5 h (assumption)
est. clicks at stake in the queue: 4,441/30d  (355 per review-hour)
   (largest scores by construction, so this is a size estimate, not a validation)

MEASURED check against the decline label (the label is not an input to the rules):


                    size  mean  random draw, same action
action                                                  
protect_page_one      10  0.40                      0.52
refresh_content       20  0.85                      0.59
rewrite_title_meta    20  0.45                      0.64
overall base rate: 0.542 | queue decline rate: 0.6 | random same-quota draw: 0.59


**Reading (a negative result that changed the playbook):**

- On one common scale the CTR rule's clicks-at-stake estimates dwarf the decline-risk ones, so a single ranked list was 100% `rewrite_title_meta`. I therefore use **quotas per action** (20 / 20 / 10) and rank **within** each action.
- Ranking `refresh_content` and `protect_page_one` by *page size* (clicks at stake) did **worse than random**: only 25% and 10% of those pages were declining versus 59% and 52% for a random draw of the same action. Large pages are stable, so size picks the wrong pages for a decline question.
- Ranking those two actions by **model probability** instead gave 85% (refresh, n = 20) and 40% (protect, n = 10) declining. I chose this ordering *after* seeing the comparison, so the 85% is optimistic and not a forecast. The `protect_page_one` slot is only 10 pages, is below the 54% base rate, and I would not rely on it: it stays human-review-only. `rewrite_title_meta` targets missed clicks, not decline, so its 45% is not a failure of that rule.
- Clicks at stake is a *size estimate* for explaining value to a stakeholder; it is not the ranking key for the two decline-based actions.

## 6. Monitoring and retrain triggers (light, non-production)

A monthly check a person can run in a notebook. Reference values come from this run's five held-out client folds.

In [8]:
ref_p50 = np.mean(fold_p50); ref_sd = np.std(fold_p50)
TRIGGERS = pd.DataFrame([
    ("ranking quality", "precision@50 on the month's reviewed/labelled pages minus the base rate",
     f"reference {ref_p50 - y.mean():+.2f} (this run); alert if below +0.10 two checks in a row"),
    ("data drift", "share of pages with avg_position = 0 (no position data)",
     f"reference {(df.avg_position == 0).mean():.1%}; alert if it moves by more than 5 percentage points"),
    ("population drift", "share of pages from clients not present in the training snapshot",
     "alert if above 20%; retrain with those clients, grouped split"),
    ("label drift", "overall decline rate in the new snapshot",
     f"reference {y.mean():.1%}; alert if it moves by more than 10 percentage points (investigate tracking change first)"),
    ("human feedback", "share of queued rows rejected by reviewers",
     "alert if above 50% in a sprint; read the rejected rows' reasons before changing rules"),
    ("calendar", "scheduled retrain", "monthly, or after any change to how trend_direction or position is computed"),
], columns=["check", "what is measured", "reference / trigger"])
print(TRIGGERS.to_string(index=False))
print(f"\nreference P@50 across folds: mean {ref_p50:.2f}, sd {ref_sd:.2f} (so one check within about +/-{2*ref_sd:.2f} is noise)")

           check                                                        what is measured                                                                                      reference / trigger
 ranking quality precision@50 on the month's reviewed/labelled pages minus the base rate                                     reference +0.29 (this run); alert if below +0.10 two checks in a row
      data drift                 share of pages with avg_position = 0 (no position data)                                       reference 4.0%; alert if it moves by more than 5 percentage points
population drift        share of pages from clients not present in the training snapshot                                            alert if above 20%; retrain with those clients, grouped split
     label drift                                overall decline rate in the new snapshot reference 54.2%; alert if it moves by more than 10 percentage points (investigate tracking change first)
  human feedback              

## 7. Exports for the paper

In [9]:
qd = df[df.action.isin(QUOTA)].copy()
qd["priority_key"] = [r[ORDER_KEY[r.action]] for _, r in qd.iterrows()]
qd["rank_in_action"] = qd.groupby("action").priority_key.rank(ascending=False, method="first").astype(int)
qd["in_sprint_queue"] = qd.apply(lambda r: r.rank_in_action <= QUOTA[r.action], axis=1)
queue = (qd.sort_values(["in_sprint_queue", "action", "rank_in_action"], ascending=[False, True, True])
         [["action", "rank_in_action", "in_sprint_queue", "content_id", "client_id", "reason_code", "clicks_at_stake_30d", "p_decline",
           "archetype", "avg_position", "impressions_90d", "clicks_90d", "ctr", "days_since_last_update", "content_age_days"]].reset_index(drop=True))
queue.insert(0, "rank", queue.index + 1)
queue.to_csv("work/outputs/action_queue.csv", index=False)
assert queue.in_sprint_queue.sum() == K
MAP.to_csv("work/outputs/archetype_action_map.csv", index=False)
TRIGGERS.to_csv("work/outputs/monitoring_triggers.csv", index=False)

metrics = {"n_pages": int(len(df)), "n_queue": int(len(queue)), "base_rate_declining": round(float(y.mean()), 4),
           "oof_auc_strict_logistic": round(float(roc_auc_score(y, p)), 4), "fold_p50": [round(float(v), 3) for v in fold_p50],
           "action_counts": df.action.value_counts().to_dict(),
           "sprint_queue_quota": QUOTA, "order_key": ORDER_KEY, "decline_rate_by_ordering": cmp_.to_dict(), "sprint_queue_clicks_at_stake_30d": round(float(topk.clicks_at_stake_30d.sum()), 1),
           "sprint_queue_decline_rate": round(float(topk.declining.mean()), 3), "random_same_quota_decline_rate": round(float(rand_decl), 3),
           "review_minutes_per_page_ASSUMPTION": REVIEW_MIN_PER_PAGE,
           "decline_rate_by_staleness": dec.decline_rate.to_dict(), "n_by_staleness": dec.n.to_dict()}
json.dump(metrics, open("work/outputs/action_playbook_metrics.json", "w"), indent=2, default=str)
print("wrote: work/outputs/action_queue.csv", f"({len(queue):,} rows), archetype_action_map.csv, monitoring_triggers.csv, action_playbook_metrics.json")
print("figure: work/figures/decay_by_staleness_and_age.png")
queue.head(10)

wrote: work/outputs/action_queue.csv (9,972 rows), archetype_action_map.csv, monitoring_triggers.csv, action_playbook_metrics.json
figure: work/figures/decay_by_staleness_and_age.png


,rank,action,rank_in_action,in_sprint_queue,content_id,client_id,reason_code,clicks_at_stake_30d,p_decline,archetype,avg_position,impressions_90d,clicks_90d,ctr,days_since_last_update,content_age_days
0,1,protect_page_one,1,True,content_4834923670c7,client_6208ef0f77,page_one_at_risk,0.223819,0.839321,young_and_volatile,8.7,1286,4,0.31,20,98
1,2,protect_page_one,2,True,content_ea307903c562,client_19581e27de,page_one_at_risk,0.276044,0.828132,young_and_volatile,3.7,1306,5,0.38,20,119
2,3,protect_page_one,3,True,content_5827e820158f,client_6208ef0f77,page_one_at_risk,0.386096,0.827349,young_and_volatile,7.6,1703,7,0.41,8,104
3,4,protect_page_one,4,True,content_849e72832462,client_19581e27de,page_one_at_risk,0.881287,0.826206,young_and_volatile,4.1,2219,16,0.72,20,95
4,5,protect_page_one,5,True,content_f34e2388fbfd,client_19581e27de,page_one_at_risk,0.274848,0.824545,young_and_volatile,8.2,1145,5,0.44,20,91
5,6,protect_page_one,6,True,content_44e4b55e8e24,client_7f2253d7e2,page_one_at_risk,0.328568,0.821421,young_and_volatile,6.7,1579,6,0.38,20,96
6,7,protect_page_one,7,True,content_754c9480298f,client_19581e27de,page_one_at_risk,1.192171,0.812844,young_and_volatile,4.9,5903,22,0.37,20,103
7,8,protect_page_one,8,True,content_403b8211b6c6,client_19581e27de,page_one_at_risk,0.593750,0.809659,young_and_volatile,6.7,2863,11,0.38,20,95
8,9,protect_page_one,9,True,content_a5f950cbaee0,client_19581e27de,page_one_at_risk,1.388132,0.800845,young_and_volatile,1.8,6076,26,0.43,20,103
9,10,protect_page_one,10,True,content_34d6a38ef80b,client_6208ef0f77,page_one_at_risk,0.960215,0.800179,young_and_volatile,2.3,2211,18,0.81,20,118


## 8. Self-check

- [x] Ranked actions with one reason code each (section 1) and archetype → action mapping (section 2)
- [x] Decay / refresh insight with n and a figure (section 3)
- [x] Intended use, limits, human-review rules, explicit no-go list (section 4)
- [x] Cost / value with the assumption labelled (section 5); monitoring and retrain triggers (section 6)
- [x] Exports written to `work/outputs/` (queue CSV stays out of git; metrics JSON committed) and figure to `work/figures/`
- [x] No client names or URLs; careful wording; non-production plan